# FedKG-Maint — Plan B4: Cross-Factory Generalization Recovery

**Azure PdM · diagnostic + controlled federated study · reproducible development experiment**

**Why B4?** Prior B1 HGB LOFO AP averaged **0.9756 (full), 0.7767 (without maintenance), 0.2442 (sensor history only)**. Prior B2 FedAvg LSTM versus MLP was **0.5777 versus 0.5737 mean AP**, but LSTM collapsed on `factory_4` (~0.23). B4 tests whether local feature drift, event-history reliance, and additional matched FL training explain this, rather than claiming any cause in advance.

**Key design rules**

- Four development factories only (`factory_1`–`factory_4`); **never load factory_5 labels, records, graphs, or predictions**. Factory 5 has been used in older studies and is not a fresh blind test.
- Reuse the **same v3 train/validation partitions, 25 feature columns, local scalers, and machine mapping**. Original v3 and B1–B3 remain read-only.
- Compare **MLP vs LSTM** with identical training endpoint samples, seeds, rounds, and model input shape. The LSTM sees a 24h chronological window, while MLP uses only the current endpoint; thus this isolates **adding temporal sequence modeling**, not parameter count.
- Feature variants: `full` (25), `no_maintenance` (maintenance channel masked to local normalized zero), `sensor_history_only` (both event-history channels and maintenance masked to zero). The network architecture and all other features remain unchanged.
- Evaluate **PR-AUC across every held-out development factory** and **early-warning performance** at predeclared **0.10 false alerts / machine-day**. The alert threshold for a held-out factory is chosen **only from the other three factories' out-of-factory validation predictions**, never from the target factory's validation labels.
- DP is **NOT applied** to newly trained B4 models. This is not an end-to-end private deployment. KG/LLM and NASA work stay out of scope until prediction architecture is selected.
- A very good score on the development folds is **not an independently validated industrial result**, and historical event features may exploit artifacts of synthetic Azure records.

**Quick start:** Run the notebook in Colab in order. Default mode `pilot` (8 FL rounds, 6,000 endpoint rows per client, seed 42); it is exploratory. For more stable evidence, set environment variable `FEDKG_B4_MODE=research` before running first cell (12 rounds, 12,000 endpoints/client, seeds 42/43/44), preferably on a GPU. To resume an interrupted run, **keep the same B_TAG and configuration** and rerun from the top; completed model results are cached with input validation. No API key needed.

In [ ]:
from pathlib import Path
import os, json, time, hashlib, random, copy, warnings, gc
import numpy as np
import pandas as pd
import joblib
from scipy.stats import ks_2samp
from sklearn.metrics import average_precision_score, roc_auc_score
warnings.filterwarnings('ignore', category=FutureWarning)
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DRIVE = Path(os.environ.get('FEDKG_DRIVE_ROOT','/content/drive/MyDrive/FedKG-Maint')).expanduser().resolve()
V3 = DRIVE/'unified_runs'/'unified_v3_final_20261025'
PREP = V3/'prep_v3'
PROC, CONF, SCALER_DIR = PREP/'processed', PREP/'configs', PREP/'scalers'
RAW = DRIVE/'data'/'raw'
PREVIOUS_B = DRIVE/'unified_runs'/'phaseB_development_v1'
MODE = os.environ.get('FEDKG_B4_MODE','pilot').strip().lower()
assert MODE in {'pilot','research','smoke'}, 'Valid modes: pilot, research, smoke'
B_TAG = os.environ.get('FEDKG_B4_TAG',f'phaseB4_generalization_{MODE}_v1')
assert B_TAG.replace('_','').replace('-','').isalnum()
OUT = DRIVE/'unified_runs'/B_TAG
assert OUT.resolve() not in {V3.resolve(), PREVIOUS_B.resolve()}
assert (V3/'RUN_COMPLETE.json').exists(), 'Missing completed unified v3 run.'
assert (PROC/'train.parquet').exists() and (PROC/'validation.parquet').exists()
assert (CONF/'feature_groups.json').exists()
assert (PREP/'manifests'/'factory_manifest.csv').exists()
assert (RAW/'PdM_failures.csv').exists()
OUT.mkdir(parents=True, exist_ok=True)
for sub in ['results','reports','predictions','logs']:(OUT/sub).mkdir(exist_ok=True)

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1<<20),b''):h.update(block)
    return h.hexdigest()
SOURCES={
    'train.parquet':PROC/'train.parquet',
    'validation.parquet':PROC/'validation.parquet',
    'feature_groups.json':CONF/'feature_groups.json',
    'factory_manifest.csv':PREP/'manifests'/'factory_manifest.csv',
    'PdM_failures.csv':RAW/'PdM_failures.csv',
}
SOURCE_HASHES={k:sha256(p) for k,p in SOURCES.items()}
print('Source v3 (read-only):',V3)
print('Source checksums:',SOURCE_HASHES)
print('Phase B4 output (separate):',OUT,'| mode:',MODE)

## B4.0 · Lock the experiment before inspecting performance

**Pilot** is a first controlled diagnostic, not the final model-choice study; **research** uses more rounds/samples and three seeds. The `smoke` mode exists solely for engineering verification and **must never be reported as evidence of utility**. The lock prevents accidental resume under different hyperparameters.

In [ ]:
import torch
from torch import nn
if MODE=='research':
    SEEDS, ROUNDS, CAP_PER_CLIENT, BATCH_SIZE = [42,43,44],12,12000,512
elif MODE=='pilot':
    SEEDS, ROUNDS, CAP_PER_CLIENT, BATCH_SIZE = [42],8,6000,512
else:
    SEEDS, ROUNDS, CAP_PER_CLIENT, BATCH_SIZE = [42],1,120,64
WINDOW=24
LR=0.002
POS_WEIGHT_CAP=30.0
ALERT_BUDGET=0.10       # max false alerts per machine-day
MIN_LEAD_HOURS=12      # actionable early warning
COOLDOWN_HOURS=24
HORIZON_HOURS=24
FEATURE_VARIANTS=['full','no_maintenance','sensor_history_only']
ARCHS=['MLP','LSTM']
DEV=[f'factory_{i}' for i in range(1,5)]
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_num_threads(min(4,max(1,os.cpu_count() or 1)))
LOCK={
    'mode':MODE,'seeds':SEEDS,'rounds':ROUNDS,'cap_per_training_client':CAP_PER_CLIENT,
    'batch_size':BATCH_SIZE,'window_hours':WINDOW,'learning_rate':LR,
    'positive_weight_cap':POS_WEIGHT_CAP,'architectures':ARCHS,'feature_variants':FEATURE_VARIANTS,
    'alert_budget_false_alarms_per_machine_day':ALERT_BUDGET,
    'minimum_advance_notice_hours':MIN_LEAD_HOURS,'cooldown_hours':COOLDOWN_HOURS,
    'source_hashes':SOURCE_HASHES,
    'selection_scope':'4-fold development LOFO only; no factory_5 data',
    'privacy':'No DP mechanism for B4 model training or prediction outputs'
}
LOCKFILE=OUT/'experiment_lock.json'
if LOCKFILE.exists():
    assert json.loads(LOCKFILE.read_text())==LOCK, 'Experiment lock changed. Set new FEDKG_B4_TAG instead of mixing studies.'
else:
    LOCKFILE.write_text(json.dumps(LOCK,indent=2))
print('Locked B4 configuration:',{k:LOCK[k] for k in ['mode','seeds','rounds','cap_per_training_client','feature_variants']})
print('CUDA available:',torch.cuda.is_available())

## B4.1 · Factory-4 distribution shift and historical-events audit

Inspect relative **machine-model composition, per-factory failure prevalence, source-event frequency, covariate distributions**, and temporal train→validation drift. KS scores summarize distribution mismatch; they are not causal explanations or significance tests here. We evaluate only development data and factory IDs 1–4.

In [ ]:
feat_cfg=json.loads((CONF/'feature_groups.json').read_text())
FEATURES=list(feat_cfg['feature_sets']['full'])
assert len(FEATURES)==25 and len(set(FEATURES))==25
SENSORS=['volt','rotate','pressure','vibration']
LABEL='label_failure_24h'
MASKED={
    'full':[],
    'no_maintenance':['hours_since_last_maintenance_log1p'],
    'sensor_history_only':['hours_since_last_maintenance_log1p','hours_since_last_error_log1p','has_prior_error']
}
assert set(sum(MASKED.values(),[]))<=set(FEATURES)
TR=pd.read_parquet(PROC/'train.parquet')
VA=pd.read_parquet(PROC/'validation.parquet')
manifest=pd.read_csv(PREP/'manifests'/'factory_manifest.csv')
fail_events=pd.read_csv(RAW/'PdM_failures.csv',parse_dates=['datetime'])[['machineID','datetime']].drop_duplicates()
for name,df in [('train',TR),('validation',VA)]:
    assert set(df.client_id.astype(str))==set(DEV),f'Unexpected client in {name}'
    assert df[['machineID','datetime']].duplicated().sum()==0
    assert set(FEATURES+[LABEL,'machineID','datetime','client_id'])<=set(df)
    assert np.isfinite(df[FEATURES].to_numpy(dtype='float64')).all()
    assert df[LABEL].isin([0,1]).all()
    df['datetime']=pd.to_datetime(df.datetime)
    df.sort_values(['machineID','datetime'],inplace=True)
    df.reset_index(drop=True,inplace=True)
assert TR.datetime.max()+pd.Timedelta(hours=24)<VA.datetime.min(),'Missing temporal purge between train and validation'
assert manifest.groupby('client_id').machineID.nunique().eq(20).all()
factory5_ids=set(manifest.loc[manifest.client_id.eq('factory_5'),'machineID'])
assert set(TR.machineID).isdisjoint(factory5_ids) and set(VA.machineID).isdisjoint(factory5_ids)
fail_events=fail_events[fail_events.machineID.isin(set(TR.machineID)|set(VA.machineID))].copy()

rows=[]
for f in DEV:
    a,b=TR[TR.client_id.eq(f)],VA[VA.client_id.eq(f)]
    models=manifest.loc[manifest.client_id.eq(f),'model'].value_counts(normalize=True)
    ff=fail_events[fail_events.machineID.isin(a.machineID.unique())]
    for model in ['model1','model2','model3','model4']:
        pass
    row={'factory':f,'machines':a.machineID.nunique(),
         'train_rows':len(a),'val_rows':len(b),
         'train_failure_24h_prevalence':float(a[LABEL].mean()),
         'val_failure_24h_prevalence':float(b[LABEL].mean()),
         'distinct_failure_events_all_source_period':len(ff)}
    row.update({f'model_mix_{m}':float(models.get(m,0)) for m in ['model1','model2','model3','model4']})
    rows.append(row)
FACTORY_PROFILE=pd.DataFrame(rows)
FACTORY_PROFILE.to_csv(OUT/'results'/'B4_factory_profiles.csv',index=False)
display(FACTORY_PROFILE)

# Samples drawn from same physical-unit feature coordinates (not cross-site local-standardized coordinates).
rng=np.random.default_rng(20261027)
shift_rows=[]
source_train={c:TR.loc[TR.client_id.eq(c)] for c in DEV}
for target in DEV:
    other=pd.concat([source_train[c] for c in DEV if c!=target],ignore_index=True)
    source=source_train[target]
    target_validation=VA.loc[VA.client_id.eq(target)]
    for kind,pop_a,pop_b in [
        ('other_factories_train_vs_target_train',other,source),
        ('target_train_vs_target_validation',source,target_validation)
    ]:
        sample_a=pop_a.sample(n=min(5000,len(pop_a)),random_state=17)
        sample_b=pop_b.sample(n=min(5000,len(pop_b)),random_state=19)
        for feature in FEATURES:
            va=sample_a[feature].to_numpy(dtype='float64')
            vb=sample_b[feature].to_numpy(dtype='float64')
            shift_rows.append({'target_factory':target,'contrast':kind,'feature':feature,
                  'ks_statistic':float(ks_2samp(va,vb,method='asymp').statistic),
                  'source_median':float(np.median(va)),'target_median':float(np.median(vb))})
SHIFT=pd.DataFrame(shift_rows)
SHIFT.to_csv(OUT/'results'/'B4_raw_feature_shift.csv',index=False)
print('Largest covariate differences for factory_4 (exploratory KS):')
display(SHIFT[SHIFT.target_factory.eq('factory_4')].sort_values('ks_statistic',ascending=False).head(15))

## B4.2 · Preprocess identical, time-safe windows

The original v3 `StandardScaler` belongs to each site. A held-out factory's scaler uses only its earlier training data, never future labels or other factories' data. The optional ablation is applied **after** local scaling: a missing channel is set to normalized `0` for every time step, meaning that network input contains no variation/information from that channel.

Validation windows never cross machine boundaries or the train/validation boundary. Both architectures score **exactly the same validation endpoints**, with source-key fingerprints saved per fold.

In [ ]:
SCALERS={f:joblib.load(SCALER_DIR/f'local_{f}.joblib') for f in DEV}
for f,sc in SCALERS.items():
    assert sc.n_features_in_==len(FEATURES),f'{f}: scaler feature count mismatch'

def locally_scaled(df):
    out=np.empty((len(df),len(FEATURES)),dtype='float32')
    for f in DEV:
        mask=df.client_id.eq(f).to_numpy()
        out[mask]=SCALERS[f].transform(df.loc[mask,FEATURES].to_numpy(dtype='float64')).astype('float32')
    assert np.isfinite(out).all()
    return out

def window_endpoints(df,window=24):
    endpoints=[]
    for mid, g in df.groupby('machineID',sort=False):
        if len(g)<window:continue
        ix=g.index.to_numpy(dtype=np.int64)
        assert ix[-1]-ix[0]+1==len(ix),f'Nonconsecutive rows {mid}'
        assert (g.datetime.diff().iloc[1:]==pd.Timedelta(hours=1)).all(),f'Irregular observations for {mid}'
        endpoints.extend(ix[window-1:])
    return np.asarray(endpoints,dtype=np.int64)

def gather_window(x,endpoints,window=24):
    ix=np.asarray(endpoints,dtype=np.int64)[:,None]-np.arange(window-1,-1,-1,dtype=np.int64)[None,:]
    return x[ix].copy()

def with_mask(x,variant):
    if not MASKED[variant]:return x
    y=x.copy()
    y[:,:, [FEATURES.index(k) for k in MASKED[variant]]]=0.0
    return y

XTR=locally_scaled(TR); XVA=locally_scaled(VA)
ENDS_TR=window_endpoints(TR,WINDOW); ENDS_VA=window_endpoints(VA,WINDOW)
assert len(ENDS_TR)>0 and len(ENDS_VA)>0
assert np.array_equal(TR.machineID.to_numpy()[ENDS_TR],TR.machineID.to_numpy()[ENDS_TR-WINDOW+1])
assert np.array_equal(VA.machineID.to_numpy()[ENDS_VA],VA.machineID.to_numpy()[ENDS_VA-WINDOW+1])
TR_Y=TR[LABEL].to_numpy(dtype='float32');VA_Y=VA[LABEL].to_numpy(dtype='int8')
END_TR_BY_SITE={f:ENDS_TR[TR.client_id.to_numpy()[ENDS_TR]==f] for f in DEV}
END_VA_BY_SITE={f:ENDS_VA[VA.client_id.to_numpy()[ENDS_VA]==f] for f in DEV}
VALID_META={f:VA.iloc[END_VA_BY_SITE[f]][['machineID','datetime']].reset_index(drop=True) for f in DEV}
VALID_LABELS={f:VA_Y[END_VA_BY_SITE[f]] for f in DEV}
VALID_HASH={f:hashlib.sha256(VALID_META[f].to_csv(index=False).encode()).hexdigest() for f in DEV}
for f in DEV:
    assert len(VALID_META[f])==len(VALID_LABELS[f]) and len(VALID_META[f])>0
    assert VALID_LABELS[f].sum()>0 and VALID_LABELS[f].sum()<len(VALID_LABELS[f]),f'Invalid label prevalence: {f}'
print('Window endpoints (train, val):',len(ENDS_TR),len(ENDS_VA))
print('Per-fold validation sample counts:',{f:len(VALID_META[f]) for f in DEV})

## B4.3 · Models, reproducible FedAvg, and automatic checkpointing

**MLP** = 25→32→8→1; **LSTM** = 25-channel × 24 hours → LSTM(32) → dense(16) → logit. Local training uses Adam with class-balanced BCE (capped positive weight), one local epoch per round, identical sampled endpoints. Client weights in FedAvg are equal because sampling caps are equal.

The experiment saves **one evaluation score vector per fold × architecture × feature group × seed** immediately after completion. A rerun with the same locked configuration resumes missing cells; it does not retrain completed cells. Saved predictions are **internal research artifacts, not differentially private**.

In [ ]:
class CompactMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model=nn.Sequential(nn.Linear(25,32),nn.ReLU(),nn.Linear(32,8),nn.ReLU(),nn.Linear(8,1))
    def forward(self,x):return self.model(x[:,-1,:]).squeeze(-1)

class TemporalLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(25,32,batch_first=True)
        self.head=nn.Sequential(nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
    def forward(self,x):return self.head(self.lstm(x)[0][:,-1,:]).squeeze(-1)

def create_model(name,seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    model=CompactMLP() if name=='MLP' else TemporalLSTM()
    return model.to(DEVICE)

@torch.no_grad()
def probabilities(model,x,batch_size=BATCH_SIZE):
    model.eval()
    chunks=[]
    for i in range(0,len(x),batch_size):
        tensor=torch.from_numpy(x[i:i+batch_size]).to(DEVICE)
        chunks.append(torch.sigmoid(model(tensor)).cpu().numpy())
    return np.concatenate(chunks).astype('float32')

def local_fit(global_model,x,y,seed,pos_weight):
    m=copy.deepcopy(global_model)
    if hasattr(m,'lstm'):m.lstm.flatten_parameters()
    m.train();opt=torch.optim.Adam(m.parameters(),lr=LR)
    crit=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight,device=DEVICE))
    rng=np.random.default_rng(seed)
    indices=rng.permutation(len(y))
    for i in range(0,len(indices),BATCH_SIZE):
        batch=indices[i:i+BATCH_SIZE]
        xx=torch.from_numpy(x[batch]).to(DEVICE)
        yy=torch.from_numpy(y[batch]).float().to(DEVICE)
        opt.zero_grad(set_to_none=True)
        loss=crit(m(xx),yy)
        assert torch.isfinite(loss),f'Non-finite loss: {seed}'
        loss.backward()
        nn.utils.clip_grad_norm_(m.parameters(),max_norm=5.)  # training stability, NOT DP
        opt.step()
    state={k:p.detach().cpu().clone() for k,p in m.state_dict().items()}
    del m
    return state

def train_federated(kind,clients,seed,pos_weight):
    model=create_model(kind,seed)
    assert len(clients)==3
    for rnd in range(ROUNDS):
        states=[local_fit(model,*clients[c],seed+10_007*rnd+97*j,pos_weight)
                for j,c in enumerate(sorted(clients))]
        new_state={k:torch.stack([state[k] for state in states]).mean(0) for k in states[0]}
        model.load_state_dict(new_state)
    if hasattr(model,'lstm'):model.lstm.flatten_parameters()
    return model

# No-data numerical self-test before expensive training.
for kind in ARCHS:
    test_x=np.random.default_rng(99).normal(size=(12,WINDOW,25)).astype('float32')
    model=create_model(kind,42)
    proba=probabilities(model,test_x)
    assert proba.shape==(12,) and np.isfinite(proba).all() and np.all((proba>=0)&(proba<=1))
print('PASS architecture shape and finite-probability tests.')

In [ ]:
# Train/evaluate every LOFO fold with *fixed* input features and sampling.
# The selected training endpoint indices are independent of model, feature variant, and seed.
ALL_RECORDS=[]
for k,held in enumerate(DEV):
    other=[f for f in DEV if f!=held]
    rng_base=20261027+101*k
    choices={}
    for j,f in enumerate(other):
        eligible=END_TR_BY_SITE[f]
        rng=np.random.default_rng(rng_base+11*j)
        choices[f]=np.sort(rng.choice(eligible,size=min(CAP_PER_CLIENT,len(eligible)),replace=False))
    sampled_labels=np.concatenate([TR_Y[choices[f]] for f in sorted(other)])
    pos=float(sampled_labels.sum());tot=len(sampled_labels)
    pw=float(min(POS_WEIGHT_CAP,max(1.,(tot-pos)/max(pos,1))))
    # Common windows used by MLP and LSTM; same score rows under each ablation.
    val_full=gather_window(XVA,END_VA_BY_SITE[held],WINDOW)
    train_full={f:(gather_window(XTR,choices[f],WINDOW),TR_Y[choices[f]]) for f in other}
    for variant in FEATURE_VARIANTS:
        val_x=with_mask(val_full,variant)
        datasets={f:(with_mask(xx,variant),yy) for f,(xx,yy) in train_full.items()}
        for kind in ARCHS:
            for seed in SEEDS:
                runid=f'{held}__{kind}__{variant}__seed{seed}'
                scorefile=OUT/'predictions'/f'{runid}.npz'
                metafile=OUT/'predictions'/f'{runid}.json'
                loaded=False
                if scorefile.exists() and metafile.exists():
                    saved=json.loads(metafile.read_text())
                    assert saved['validation_rows_sha256']==VALID_HASH[held]
                    assert saved['experiment_lock_sha256']==sha256(LOCKFILE)
                    assert saved['fold']==held and saved['variant']==variant and saved['architecture']==kind and saved['seed']==seed
                    with np.load(scorefile) as a:scores=a['scores']
                    assert scores.shape==(len(VALID_LABELS[held]),) and np.isfinite(scores).all()
                    loaded=True
                if not loaded:
                    started=time.time()
                    network=train_federated(kind,datasets,seed,pw)
                    scores=probabilities(network,val_x)
                    del network
                    assert scores.shape==(len(VALID_LABELS[held]),) and np.isfinite(scores).all()
                    np.savez_compressed(scorefile,scores=scores)
                    meta={'fold':held,'variant':variant,'architecture':kind,'seed':seed,
                          'validation_rows_sha256':VALID_HASH[held],
                          'experiment_lock_sha256':sha256(LOCKFILE),
                          'n_train':tot,'positive_weight':pw,'elapsed_seconds':round(time.time()-started,2),
                          'privacy':'NON-DP INTERNAL RESEARCH PREDICTIONS'}
                    metafile.write_text(json.dumps(meta,indent=2))
                    print('TRAINED ',runid,'AP',round(float(average_precision_score(VALID_LABELS[held],scores)),4))
                else:
                    print('RESTORED',runid)
                ALL_RECORDS.append({
                    'fold':held,'architecture':kind,'variant':variant,'seed':seed,
                    'rounds':ROUNDS,'train_rows':tot,'validation_rows':len(scores),
                    'validation_rows_sha256':VALID_HASH[held],
                    'prevalence':float(VALID_LABELS[held].mean()),
                    'pr_auc':float(average_precision_score(VALID_LABELS[held],scores)),
                    'roc_auc':float(roc_auc_score(VALID_LABELS[held],scores)),
                    'prediction_path':str(scorefile)
                })
                pd.DataFrame(ALL_RECORDS).to_csv(OUT/'results'/'B4_model_fold_metrics.csv',index=False)
        del val_x,datasets
    del val_full,train_full
    gc.collect()
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
MODEL_ROWS=pd.DataFrame(ALL_RECORDS)
expected=len(DEV)*len(ARCHS)*len(FEATURE_VARIANTS)*len(SEEDS)
assert len(MODEL_ROWS)==expected and not MODEL_ROWS.duplicated(['fold','architecture','variant','seed']).any()
MODEL_SUMMARY=MODEL_ROWS.groupby(['variant','architecture'],as_index=False).agg(
    mean_ap=('pr_auc','mean'),worst_fold_ap=('pr_auc','min'),
    sd_ap=('pr_auc','std'),mean_roc_auc=('roc_auc','mean'))
MODEL_SUMMARY['robust_score']=0.6*MODEL_SUMMARY['mean_ap']+0.4*MODEL_SUMMARY['worst_fold_ap']
MODEL_SUMMARY.to_csv(OUT/'results'/'B4_model_summary.csv',index=False)
display(MODEL_SUMMARY.sort_values('robust_score',ascending=False))

## B4.4 · Calibrate alerts at **the same false-alarm budget**

**Procedure (cross-fold development thresholding):** For each architecture/variant/seed and target factory, collect predictions and labels on the *other three factories' validation folds*. Choose the threshold that offers the highest 12-hour actionable event recall **subject to** ≤0.10 false alerts per machine-day on those other folds. Then evaluate that frozen threshold on the target fold. We never look at the target fold's labels when choosing its own threshold.

This is a useful **cross-factory diagnostic**, but model training datasets overlap between LOFO folds; therefore the 4-fold threshold selection is not a fully independent nested statistical test. If no threshold can meet useful recall at the budget, the routine may select a no-alert threshold, and the report will say so. Thresholds are not calibrated probabilities.

In [ ]:
# Efficient event matcher; the first subsequent event after an alarm is authoritative.
def operational_metrics(meta,scores,events,threshold,
                        minimum_lead_h=MIN_LEAD_HOURS,cooldown_h=COOLDOWN_HOURS,horizon_h=HORIZON_HOURS):
    assert len(meta)==len(scores)
    allowed=pd.Timedelta(hours=horizon_h)
    min_lead=pd.Timedelta(hours=minimum_lead_h)
    cooldown=pd.Timedelta(hours=cooldown_h)
    score=np.asarray(scores,dtype='float64')
    assert np.isfinite(score).all()
    total=true=0; matched_events=set();eligible_events=set();leads=[]
    for mid,idx in meta.groupby('machineID',sort=False).indices.items():
        g=meta.iloc[idx]
        ts=pd.to_datetime(g.datetime).to_numpy(dtype='datetime64[ns]')
        pred=score[idx]
        assert (np.diff(ts)>np.timedelta64(0,'ns')).all()
        ff=np.sort(events.loc[events.machineID.eq(mid),'datetime'].to_numpy(dtype='datetime64[ns]'))
        # Evaluate only failures for which a valid lead-time alarm could exist in this fold.
        for ft in ff:
            left=ft-np.timedelta64(horizon_h,'h')
            right=ft-np.timedelta64(minimum_lead_h,'h')
            lo=np.searchsorted(ts,left,side='left')
            hi=np.searchsorted(ts,right,side='right')
            if hi>lo:eligible_events.add((str(mid),str(pd.Timestamp(ft))))
        candidates=np.flatnonzero(pred>=threshold)
        next_allowed=np.datetime64('1900-01-01','ns')
        for j in candidates:
            t=ts[j]
            if t<next_allowed:continue
            total+=1
            next_allowed=t+cooldown.to_timedelta64()
            index=np.searchsorted(ff,t,side='right')
            if index<len(ff):
                lead=(ff[index]-t)/np.timedelta64(1,'h')
                if minimum_lead_h<=lead<=horizon_h:
                    true+=1
                    matched_events.add((str(mid),str(pd.Timestamp(ff[index]))))
                    leads.append(float(lead))
    denom_days=len(meta)/24.0
    assert 0<=true<=total
    return {'threshold':float(threshold),'total_alerts':int(total),
            'true_alerts':int(true),'false_alerts':int(total-true),
            'eligible_events':len(eligible_events),
            'detected_events':len(matched_events & eligible_events),
            'machine_days':float(denom_days),
            'event_recall':len(matched_events & eligible_events)/len(eligible_events) if eligible_events else np.nan,
            'actionable_alert_precision':true/total if total else 0.0,
            'false_alarms_per_machine_day':(total-true)/denom_days if denom_days else np.nan,
            'median_true_lead_hours':float(np.median(leads)) if leads else np.nan}

# Self-test with event exactly 12h ahead of alarm.
dummy_meta=pd.DataFrame({'machineID':[777]*48,'datetime':pd.date_range('2015-01-01',periods=48,freq='h')})
dummy_events=pd.DataFrame({'machineID':[777],'datetime':[pd.Timestamp('2015-01-01 12:00:00')]})
dummy=np.zeros(48,dtype='float32');dummy[0]=.9
assert operational_metrics(dummy_meta,dummy,dummy_events,.5,minimum_lead_h=12)['detected_events']==1
assert operational_metrics(dummy_meta,dummy,dummy_events,.5,minimum_lead_h=13)['detected_events']==0
print('PASS operational event-matching tests.')

# Pull predictions from locked experiment; cannot draw Factory 5 data.
def get_predictions(fold,kind,variant,seed):
    runid=f'{fold}__{kind}__{variant}__seed{seed}'
    with np.load(OUT/'predictions'/f'{runid}.npz') as a:p=a['scores']
    assert len(p)==len(VALID_META[fold])
    return p

def pooled_rates(folds,pred_map,threshold):
    metrics=[operational_metrics(VALID_META[f],pred_map[f],fail_events,threshold) for f in folds]
    false=sum(x['false_alerts'] for x in metrics)
    days=sum(x['machine_days'] for x in metrics)
    eligible=sum(x['eligible_events'] for x in metrics)
    detected=sum(x['detected_events'] for x in metrics)
    return {'far':false/days if days else np.nan,
            'event_recall':detected/eligible if eligible else 0.0,
            'detected':detected,'eligible':eligible,
            'false_alerts':false,'days':days}

OPS_ROWS=[]
for variant in FEATURE_VARIANTS:
    for kind in ARCHS:
        for seed in SEEDS:
            scores_by_fold={f:get_predictions(f,kind,variant,seed) for f in DEV}
            for held in DEV:
                calibration=[f for f in DEV if f!=held]
                calibration_scores=np.concatenate([scores_by_fold[f] for f in calibration])
                # Same predefined set of quantile fractions per model; includes no-alert option.
                q=np.r_[0,.2,.4,.55,.7,.8,.86,.9,.93,.95,.97,.98,.99,.995,.999,1.0]
                thresholds=np.unique(np.r_[np.quantile(calibration_scores,q),
                                            np.nextafter(calibration_scores.max(),np.inf)])
                candidates=[]
                for threshold in thresholds:
                    z=pooled_rates(calibration,scores_by_fold,float(threshold))
                    candidates.append({'threshold':float(threshold),**z})
                feasible=[r for r in candidates if r['far']<=ALERT_BUDGET+1e-12]
                assert feasible, 'No-alert threshold must always be feasible.'
                # Max development event recall; tie-break with fewer false alarms.
                best=sorted(feasible,key=lambda r:(-r['event_recall'],r['far'],-r['threshold']))[0]
                observed=operational_metrics(VALID_META[held],scores_by_fold[held],fail_events,best['threshold'])
                OPS_ROWS.append({
                    'heldout_factory':held,'architecture':kind,'variant':variant,'seed':seed,
                    'selected_threshold_other_folds':best['threshold'],
                    'calibration_false_alarms_per_machine_day':best['far'],
                    'calibration_event_recall':best['event_recall'],
                    **{f'heldout_{k}':v for k,v in observed.items() if k!='threshold'}
                })
OPS=pd.DataFrame(OPS_ROWS)
OPS.to_csv(OUT/'results'/'B4_cross_fold_alert_metrics.csv',index=False)
OPS_SUMMARY=OPS.groupby(['variant','architecture'],as_index=False).agg(
    mean_test_far=('heldout_false_alarms_per_machine_day','mean'),
    mean_test_event_recall=('heldout_event_recall','mean'),
    min_test_event_recall=('heldout_event_recall','min'),
    total_detected_events=('heldout_detected_events','sum'),
    total_eligible_events=('heldout_eligible_events','sum'),
    total_false_alerts=('heldout_false_alerts','sum'),
    total_machine_days=('heldout_machine_days','sum'))
OPS_SUMMARY['pooled_event_recall']=OPS_SUMMARY.total_detected_events/OPS_SUMMARY.total_eligible_events.replace(0,np.nan)
OPS_SUMMARY['pooled_far']=OPS_SUMMARY.total_false_alerts/OPS_SUMMARY.total_machine_days
OPS_SUMMARY.to_csv(OUT/'results'/'B4_operational_summary.csv',index=False)
display(OPS_SUMMARY.sort_values('pooled_event_recall',ascending=False))

## B4.5 · Paired diagnosis, model-choice recommendation, and release audit

**Decision priorities**: (1) Robustness across the four development factories, especially factory 4, (2) event-level detection at a comparable false-alert budget, and (3) mean PR-AUC. A new architecture does **not** replace the existing frozen v3 model based on this development experiment alone.

This notebook writes the final tables and a concise `.md`/`.json` report. No old run files are changed. A `B4_COMPLETE.json` marker means **the code has completed**, not that an operational utility or privacy threshold was passed.

In [ ]:
# Pair MLP/LSTM on exact same validation endpoints, fold, feature mask and seed.
paired=MODEL_ROWS.pivot(index=['fold','variant','seed'],columns='architecture',values='pr_auc')
assert not paired.isna().any().any()
paired['LSTM_minus_MLP']=paired['LSTM']-paired['MLP']
paired.reset_index().to_csv(OUT/'results'/'B4_paired_LSTM_minus_MLP.csv',index=False)

# Do not run when original inputs were modified.
assert all(sha256(p)==SOURCE_HASHES[name] for name,p in SOURCES.items()),'Source data changed during run!'
# Validate that historical held-out factory never entered source input data.
assert 'factory_5' not in set(TR.client_id)|set(VA.client_id)
assert set(OPS.heldout_factory)==set(DEV)
assert set(MODEL_ROWS.fold)==set(DEV)
assert len(MODEL_ROWS)==len(DEV)*len(ARCHS)*len(FEATURE_VARIANTS)*len(SEEDS)

RANKING=MODEL_SUMMARY.merge(OPS_SUMMARY,on=['variant','architecture'],validate='one_to_one')
# Descriptive benchmark only: a rank isn't external held-out validation.
RANKING=RANKING.sort_values(['worst_fold_ap','mean_ap','pooled_event_recall'],ascending=False)
RANKING.to_csv(OUT/'results'/'B4_ranked_dev_candidates.csv',index=False)

diagnostic={
  'phase':'B4 exploratory cross-factory development only',
  'input_source_v3':str(V3),'configuration':LOCK,
  'files_unchanged':True,'factory_5_used':False,
  'all_model_runs_completed':len(MODEL_ROWS),
  'model_summary':MODEL_SUMMARY.to_dict('records'),
  'operational_summary':OPS_SUMMARY.to_dict('records'),
  'paired_diff_per_feature_variant':paired.reset_index().groupby('variant')['LSTM_minus_MLP'].agg(['mean','min','max']).reset_index().to_dict('records'),
  'candidate_ranked_by_worst_site_first':RANKING[['architecture','variant','worst_fold_ap','mean_ap','pooled_event_recall','pooled_far']].to_dict('records'),
  'chosen_final_model':None,
  'not_claimed':['Fresh independent factory_5 test','End-to-end DP','Any cryptographic secure aggregation',
                 'NASA performance','LLM benefits','Causal link between maintenance and failure'],
  'next_step':'Inspect factory_4 degradation and cross-fold FAR; lock a candidate on development, then compare matched DP/non-DP and validate on new independent test data.'
}
(OUT/'reports'/'B4_generalization_decision.json').write_text(json.dumps(diagnostic,indent=2,default=str))

md_lines=[
  '# FedKG-Maint · B4 Cross-Factory Generalization Report','',
  f'**Experiment mode:** {MODE} · **Model runs:** {len(MODEL_ROWS)}',
  '**Factory 5:** NOT read or used. **Privacy:** new models are non-DP.','',
  '## Model performance (development LOFO)',
  MODEL_SUMMARY.to_markdown(index=False,floatfmt='.4f'),'',
  '## Paired improvement (LSTM minus MLP)',
  paired.reset_index().to_markdown(index=False,floatfmt='.4f'),'',
  '## Event recall at cross-fold budgeted false alarms',
  OPS_SUMMARY.to_markdown(index=False,floatfmt='.4f'),'',
  '## Factory profiles',FACTORY_PROFILE.to_markdown(index=False,floatfmt='.4f'),'',
  '## Limitations',
  '- All evaluation is development-only; no new independent test site is evaluated.',
  '- Candidate comparison uses shared validation data across model selection and reporting; outcome is exploratory.',
  '- Thresholds are cross-fold but folds use overlapping training datasets.',
  '- This study does not supply any new differential privacy guarantee.',
  '- Pilot/smoke mode is not final research-grade seed-stability evidence.',
  '- Historical event features may encode synthetic data-generating patterns.',
  '- No model is automatically released or marked as a winner.',
]
(OUT/'reports'/'B4_generalization_decision.md').write_text('\n'.join(md_lines))
(OUT/'B4_COMPLETE.json').write_text(json.dumps({
    'mode':MODE,'completed':True,'source_hashes_verified':True,'all_models':len(MODEL_ROWS),
    'decision_report':'reports/B4_generalization_decision.md'},indent=2))
print('B4 COMPLETE — CODE EXECUTION ONLY (not accuracy/privacy certification)')
print('See',OUT/'reports'/'B4_generalization_decision.md')
print('Top-ranked development candidate (not selected for deployment):')
display(RANKING[['architecture','variant','worst_fold_ap','mean_ap','pooled_event_recall','pooled_far']].head(6))